# Random Feature Ablation Control

Tests whether ablating random features (norm-matched to Feature 1215) also causes gender-agreement errors, or whether the effect is specific to Feature 1215.

**Protocol:**
1. Measure the norm (mean magnitude) of Feature 1215 in the PT corpus
2. Select 10 random features with similar norm (±20%)
3. Ablate each one with the same 4 prompts × 5 seeds = 20 generations
4. Automatically classify gender-agreement errors
5. Compare: Feature 1215 → 20/20 errors. Random features → ?/20

**Runtime:** ~15 min on T4 | **Output:** `exp_random_ablation_results.json`

In [ ]:
!pip install transformer_lens sae_lens -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "./data/checkpoints/"
import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import torch
import numpy as np
import json
import re
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"

from sae_lens import SAE, HookedSAETransformer

print("Loading model...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b", device=device, dtype=torch.float16,
)
tokenizer = model.tokenizer

print("Loading SAE...")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
print(f"SAE: {sae.cfg.d_sae} features")

## Step 1: Measure the norm of Feature 1215 and select control features

In [ ]:
TARGET_FEATURE = 1215
N_CONTROL = 10
TOLERANCE = 0.20

total_act = None
stats_path = os.path.join(SAVE_DIR, "stats_layer13.pt")
if os.path.exists(stats_path):
    print("Loading precomputed stats...")
    ckpt = torch.load(stats_path, weights_only=False)
    print(f"Available keys: {list(ckpt.keys())}")

    # Tentar diferentes formatos de chave
    if "count_wiki_pt" in ckpt:
        total_act = ckpt["count_wiki_pt"] + ckpt.get("count_mc4_pt", 0)
    elif "freq_pt" in ckpt:
        total_act = ckpt["freq_pt"]  # already a frequency, use as activity proxy
    elif "count_pt" in ckpt:
        total_act = ckpt["count_pt"]

    if total_act is not None:
        print(f"Feature 1215: atividade={total_act[TARGET_FEATURE]:.4f}")
    else:
        print("Unrecognized stats format — filtering by decoder norm only.")
else:
    print("Stats not found. Filtering by decoder norm only.")

decoder_norms = sae.W_dec.data.norm(dim=1).cpu()
target_norm = decoder_norms[TARGET_FEATURE].item()
print(f"\nFeature 1215 decoder norm: {target_norm:.4f}")

norm_lo = target_norm * (1 - TOLERANCE)
norm_hi = target_norm * (1 + TOLERANCE)
print(f"Range for matching: [{norm_lo:.4f}, {norm_hi:.4f}]")

candidates = []
for fid in range(sae.cfg.d_sae):
    if fid == TARGET_FEATURE:
        continue
    n = decoder_norms[fid].item()
    if norm_lo <= n <= norm_hi:
        if total_act is not None and total_act[fid] < 100:
            continue
        candidates.append((fid, n))

print(f"Features com norm similar: {len(candidates)}")

np.random.seed(42)
selected_idx = np.random.choice(len(candidates), size=min(N_CONTROL, len(candidates)), replace=False)
control_features = [candidates[i] for i in selected_idx]

print(f"\nSelected control features:")
print(f"{'Feature':<10} {'Norm':<10} {'Ratio vs 1215':<15}")
print("-" * 35)
for fid, n in control_features:
    print(f"{fid:<10} {n:<10.4f} {n/target_norm:<15.3f}")

CONTROL_FEATURE_IDS = [fid for fid, _ in control_features]

## Step 2: Generation with ablation (Feature 1215 + 10 controls × 4 prompts × 5 seeds)

In [ ]:
PROMPTS = [
    "A menina bonita foi à",
    "A diretora da empresa apresentou a nova",
    "A professora explicou que a questão",
    "A médica recomendou que a paciente",
]

SEEDS = [42, 123, 456, 789, 1337]

def generate_with_ablation(model, sae, tokenizer, prompt, feature_id,
                           max_new_tokens=60, temperature=0.7,
                           hook_name=HOOK_NAME, seed=42):
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    fid_t = torch.tensor([feature_id], device=device)

    def ablation_hook(value, hook):
        with torch.no_grad():
            sae_acts = sae.encode(value)
            modified = sae_acts.clone()
            modified[:, :, fid_t] = 0.0
            reconstructed = sae.decode(modified)
            error = value - sae.decode(sae_acts)
            return reconstructed + error

    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model.run_with_hooks(
                generated, fwd_hooks=[(hook_name, ablation_hook)],
            )
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


def generate_baseline(model, tokenizer, prompt, max_new_tokens=60,
                      temperature=0.7, seed=42):
    torch.manual_seed(seed)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(generated)
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            if next_token.item() == tokenizer.eos_token_id:
                break
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=-1)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


GENDER_ERROR_PATTERNS = [
    (r'\b[Aa]\s+(?:bolo|carro|menino|gato|cachorro|homem|pai|filho|marido|irmão|tio|avô|primo|neto|rapaz|senhor|professor|diretor|médico|doutor|novo|velho|bonito|grande|pequeno|alto|baixo|gordo|magro|jovem|rico|pobre|primeiro|segundo|terceiro|último|bom|mau|meu|seu|nosso|dono|consumo|avanço|comércio|veículo|banheiro|quarto|trabalho|emprego|dinheiro|mercado|governo|país|mundo|tempo|momento|lugar|caso|problema|resultado|processo|projeto|sistema|modelo|programa|serviço|produto|estudo|grupo|número)\b', 'fem_art_masc_noun'),
    (r'\b[Oo]\s+(?:menina|mulher|mãe|filha|esposa|irmã|tia|avó|prima|neta|moça|senhora|professora|diretora|médica|doutora|nova|velha|bonita|grande|pequena|alta|baixa|gorda|magra|jovem|rica|pobre|primeira|segunda|terceira|última|boa|má|minha|sua|nossa|dona)\b', 'masc_art_fem_noun'),
    (r'\b(?:da|na|pela|uma)\s+(?:bolo|carro|menino|gato|cachorro|homem|pai|filho|marido|irmão|novo|velho|bonito|veículo|banheiro|consumo|avanço|comércio|modelo|câncer|animal)\b', 'fem_prep_masc_noun'),
]

def count_gender_errors(text):
    errors = []
    for pattern, error_type in GENDER_ERROR_PATTERNS:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for m in matches:
            errors.append({"type": error_type, "match": m if isinstance(m, str) else m[0]})
    return errors


print("Starting generation...")
all_features = [TARGET_FEATURE] + CONTROL_FEATURE_IDS
results = {"target_feature": TARGET_FEATURE, "control_features": CONTROL_FEATURE_IDS,
           "prompts": PROMPTS, "seeds": SEEDS, "per_feature": {}}

t0 = time.time()

for feat_idx, fid in enumerate(all_features):
    label = "TARGET (1215)" if fid == TARGET_FEATURE else f"CONTROL ({fid})"
    print(f"\n{'='*60}")
    print(f"Feature {feat_idx+1}/{len(all_features)}: {label}")

    feat_results = {"feature_id": fid, "is_target": fid == TARGET_FEATURE,
                    "decoder_norm": decoder_norms[fid].item(), "generations": []}

    for prompt in PROMPTS:
        for seed in SEEDS:
            ablated = generate_with_ablation(model, sae, tokenizer, prompt, fid,
                                            seed=seed)
            baseline = generate_baseline(model, tokenizer, prompt, seed=seed)

            abl_errors = count_gender_errors(ablated)
            base_errors = count_gender_errors(baseline)

            feat_results["generations"].append({
                "prompt": prompt, "seed": seed,
                "baseline": baseline, "ablated": ablated,
                "baseline_errors": len(base_errors),
                "ablated_errors": len(abl_errors),
                "error_details": [e["match"] for e in abl_errors],
            })

    total_abl_errors = sum(g["ablated_errors"] for g in feat_results["generations"])
    total_base_errors = sum(g["baseline_errors"] for g in feat_results["generations"])
    n_with_errors = sum(1 for g in feat_results["generations"] if g["ablated_errors"] > 0)
    feat_results["summary"] = {
        "total_ablated_errors": total_abl_errors,
        "total_baseline_errors": total_base_errors,
        "n_generations_with_errors": n_with_errors,
        "n_total_generations": len(feat_results["generations"]),
        "error_rate": n_with_errors / len(feat_results["generations"]),
    }

    results["per_feature"][str(fid)] = feat_results
    print(f"  Ablation errors: {total_abl_errors} | Baseline: {total_base_errors} | "
          f"Gerações com erro: {n_with_errors}/20")

    elapsed = time.time() - t0
    remaining = elapsed / (feat_idx + 1) * (len(all_features) - feat_idx - 1)
    print(f"  Time: {elapsed:.0f}s | ~{remaining:.0f}s remaining")

print(f"\nDone in {time.time() - t0:.0f}s")

## Step 3: Analysis and saving

In [ ]:
print("=" * 70)
print("RESULTS: RANDOM ABLATION vs FEATURE 1215 (GENDER)")
print("=" * 70)

target_data = results["per_feature"][str(TARGET_FEATURE)]
print(f"\n{'Feature':<12} {'Norm':<8} {'Target?':<8} {'Abl Errors':<10} {'Base Errors':<10} "
      f"{'Gens c/ erro':<14} {'Taxa erro':<10}")
print("-" * 80)

print(f"{TARGET_FEATURE:<12} {target_data['decoder_norm']:<8.4f} {'SIM':<8} "
      f"{target_data['summary']['total_ablated_errors']:<10} "
      f"{target_data['summary']['total_baseline_errors']:<10} "
      f"{target_data['summary']['n_generations_with_errors']}/20{'':>6} "
      f"{target_data['summary']['error_rate']:.1%}")

control_error_rates = []
for fid in CONTROL_FEATURE_IDS:
    d = results["per_feature"][str(fid)]
    s = d["summary"]
    control_error_rates.append(s["error_rate"])
    print(f"{fid:<12} {d['decoder_norm']:<8.4f} {'NO':<8} "
          f"{s['total_ablated_errors']:<10} {s['total_baseline_errors']:<10} "
          f"{s['n_generations_with_errors']}/20{'':>6} {s['error_rate']:.1%}")

print(f"\n{'='*70}")
target_rate = target_data["summary"]["error_rate"]
mean_control = np.mean(control_error_rates)
max_control = np.max(control_error_rates)

print(f"Feature 1215 (target): {target_rate:.1%} generations with gender error")
print(f"Random controls: mean = {mean_control:.1%}, max = {max_control:.1%}")
print(f"target/control ratio: {target_rate / (mean_control + 1e-10):.1f}×")

if target_rate > max_control * 2:
    print("\n✓ Feature 1215 causes SIGNIFICANTLY more gender errors "
          "que qualquer feature aleatória de norma similar.")
elif target_rate > mean_control * 2:
    print("\n~ Feature 1215 causes more errors than the average, but at least "
          "uma feature aleatória também causa erros substanciais.")
else:
    print("\n✗ The effect may be attributed to general perturbation, "
          "não à especificidade de gênero da Feature 1215.")

output = {
    "experiment": "random_ablation_control",
    "target_feature": TARGET_FEATURE,
    "n_control_features": len(CONTROL_FEATURE_IDS),
    "control_feature_ids": CONTROL_FEATURE_IDS,
    "n_prompts": len(PROMPTS),
    "n_seeds": len(SEEDS),
    "n_generations_per_feature": len(PROMPTS) * len(SEEDS),
    "target_error_rate": target_rate,
    "control_mean_error_rate": mean_control,
    "control_max_error_rate": max_control,
    "control_error_rates": {str(fid): r for fid, r in zip(CONTROL_FEATURE_IDS, control_error_rates)},
    "per_feature_details": {
        str(fid): results["per_feature"][str(fid)]["summary"]
        for fid in [TARGET_FEATURE] + CONTROL_FEATURE_IDS
    },
}

output_path = os.path.join(SAVE_DIR, "exp_random_ablation_results.json")
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f"\nResults saved: {output_path}")

full_path = os.path.join(SAVE_DIR, "exp_random_ablation_full.json")
with open(full_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Dados completos: {full_path}")